Haan. Agar **professor ko starting se end tak practically dikhana hai**, to ab tumhare paas ek working pipeline hai. Main ise exactly **demo workflow** ki tarah rakhunga — extra Vicon/analysis abhi nahi.

### Professor ko ye complete flow dikhao

```text
1. Docker Desktop start
        ↓
2. Project folder open
        ↓
3. Docker container start
        ↓
4. MEND-G01 ON
        ↓
5. Windows Bluetooth detects IMU
        ↓
6. Windows Python + Bleak connects to IMU
        ↓
7. BLE notifications receive
        ↓
8. Data TCP :5555 se Docker ko
        ↓
9. Docker TrialDataLogger data process karta hai
        ↓
10. CSV generate hoti hai
        ↓
11. CSV me IMU values + timestamps verify
        ↓
12. Sampling rate verify (~42 Hz)
```

## 1. Professor ke saamne starting point

Project folder:

```powershell
cd "C:\Users\parwa\Downloads\ble_data_collector\ble_data_collector"
```

Check:

```powershell
dir
```

Tumhare project me roughly ye important files honi chahiye:

```text
docker-compose.yml
Dockerfile
launch.sh
src\
    main.py
    data_collector.py
    ble_core.py
windows_ble_bridge.py
docker_bridge_receiver.py
docker-compose.windows-ble.yml
data\
```

---

## 2. Docker start karo

Agar Docker Desktop already running hai:

```powershell
docker version
```

Agar daemon response aa raha hai → ✅

Agar Docker Desktop closed hai:

```powershell
Start-Process "$env:LOCALAPPDATA\Programs\DockerDesktop\Docker Desktop.exe"
```

Phir:

```powershell
docker version
```

---

## 3. MEND-G01 ON karo

Professor ko simply batao:

> "The MEND-G01 IMU communicates with the laptop through BLE."

Windows Bluetooth ON hona chahiye.

**Important:** IMU ko kisi doosre BLE program se simultaneously connect mat karna.

---

# 4. Docker receiver start karo

### Terminal 1

```powershell
docker compose -f docker-compose.yml -f docker-compose.windows-ble.yml run --rm --service-ports ble_data_collector
```

Successful hone par ye dikhega:

```text
Docker Windows-BLE receiver READY
Listening on 0.0.0.0:5555
Output device: right/foot
CSV will be written by the existing TrialDataLogger.
Now start windows_ble_bridge.py on Windows.
```

### Yahan pause.

Professor ko explain:

> "Because Docker Desktop is running through Windows/WSL2, the container cannot directly access the Windows Bluetooth controller. So I use a Windows-native BLE bridge and send the received packets to the Docker container over TCP."

**Ye actually tumhare current setup ka important technical point hai.**

---

# 5. Windows BLE bridge start karo

### Terminal 2

Project folder me:

```powershell
python windows_ble_bridge.py
```

Expected:

```text
Connecting to MEND-G01: FD:69:7C:87:0C:26
Connected to Docker receiver at 127.0.0.1:5555
Connecting to MEND-G01...
Connected: True
Notifications enabled.
Sending init command: F

READY. Collecting BLE packets...
Press CTRL+C to stop.
```

Phir:

```text
packets=...
rate=42.2 Hz
```

aisa continuously aayega.

### Professor ko bolo:

> "The Windows side is receiving the BLE notifications at approximately 42 Hz."

Report me effective logging approximately **40 Hz** diya gaya hai, so tumhara observed ~42 Hz uske close hai.

---

# 6. Short actual experiment karo

Ab IMU ko stationary rakho initially.

For example:

```text
0–5 sec     → stationary
5–15 sec    → move IMU
15–20 sec   → stationary
```

Isse baad me raw signal me movement clearly identify kar sakte ho.

**Abhi walking/gait analysis karna zaroori nahi hai.**

20–30 seconds ka demo enough hai.

---

# 7. Recording stop karo

Windows bridge wale terminal me:

```text
CTRL + C
```

Docker receiver ko bhi:

```text
CTRL + C
```

Isse CSV flush ho jayegi.

---

# 8. CSV verify karo

```powershell
Get-ChildItem .\data\*.csv |
Sort-Object LastWriteTime -Descending |
Select-Object -First 5 FullName,Length,LastWriteTime
```

Tumhare case me new file:

```text
data\trial_001_right.csv
```

~4.45 MB thi, so clearly data aa raha hai.

---

# 9. CSV ka actual data professor ko dikhao

```powershell
Get-Content ".\data\trial_001_right.csv" -TotalCount 5
```

Header me tumhare paas:

```text
timestamp
packet_time
packet_index
device_key
side
segment
qw
qx
qy
qz
acc_x
acc_y
acc_z
gyr_x
gyr_y
filtered_gyr_y
gyr_z
calibration_phase
```

Professor ko explain:

### Orientation

```text
qw qx qy qz
```

### Acceleration

```text
acc_x acc_y acc_z
```

### Angular velocity

```text
gyr_x gyr_y gyr_z
```

### Timing

```text
timestamp
packet_time
packet_index
```

---

# 10. Sampling rate bhi prove karo

Ye **bahut important** hai.

Sirf:

```text
packets=... rate=42.3 Hz
```

dikhana enough hai, but better hai CSV se bhi calculate karna.

PowerShell me:

```powershell
python -c "import pandas as pd; f='data/trial_001_right.csv'; d=pd.read_csv(f); dt=d['timestamp'].diff().dropna(); print('Rows:',len(d)); print('Duration:',d['timestamp'].iloc[-1]-d['timestamp'].iloc[0],'sec'); print('Mean sampling rate:',1/dt.mean(),'Hz')"
```

Isse tum professor ko **actual CSV-derived sampling rate** dikha sakte ho.

---

# 11. Professor ko architecture ek baar diagram me explain karo

Notebook/PowerPoint me ye simple diagram bana do:

```text
       MEND-G01 IMU
             │
             │ BLE
             ▼
     Windows Bluetooth
             │
             ▼
       Python + Bleak
       Windows Bridge
             │
             │ TCP :5555
             ▼
      Docker Container
             │
             ▼
     TrialDataLogger
             │
             ▼
       CSV File
             │
             ▼
   IMU Raw Sensor Dataset
```

### And explain:

> "I first established the BLE connection with the MEND-G01 on Windows. Since Docker Desktop/WSL2 does not directly expose the Windows Bluetooth controller to the Linux container, I separated BLE acquisition from Docker processing. The Windows Python bridge receives the BLE notifications and forwards them through TCP to the Docker container, where the existing data logger converts the packets and stores them as CSV."

---

# 12. Report ke context me tumne kya complete kiya?

Report ke acquisition portion me essentially:

**IMU → BLE → laptop → timestamped data → CSV**

wala part demonstrate ho gaya.

Tumhari current implementation me additionally:

**Windows BLE → TCP → Docker**

bridge add hua hai because of your Windows + Docker environment.

---

## ⚠️ Abhi professor ko kya **NAHI** dikhana

Abhi unnecessarily ye sab start mat karna:

❌ Vicon
❌ MediaPipe
❌ gait event detection
❌ Heel Strike / Toe Off
❌ ZUPT
❌ double integration
❌ filtering experiments
❌ deep learning
❌ validation

Pehle professor ko **raw IMU acquisition successfully working** dikhao.

---

# 🎯 Tumhara 10-minute professor demo

Agar sirf 10 minute mile:

**1 min:** MEND-G01 + architecture explain
**2 min:** Docker receiver start
**2 min:** Windows BLE bridge start
**2 min:** ~42 Hz live packet reception show
**1 min:** CSV open/show
**2 min:** fields + sampling rate explain

Bas.

### Final outcome

Professor ke saamne tum ye prove kar paoge:

> **"MEND-G01 se BLE data successfully receive ho raha hai, approximately 42 Hz par packets aa rahe hain, Docker-based pipeline un packets ko receive/process kar rahi hai, aur timestamped IMU data CSV me successfully save ho raha hai."**

**Ye tumhara Phase 1 / raw IMU data acquisition ka solid demo hai.**
